In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

df = pd.read_pickle('../data/interim/rural_cleaned.pkl')
print("Shape:", df.shape)
df.dtypes

Shape: (3371, 13)


v190a                         int8
v106                          int8
v133                          int8
v116                          int8
v113                          int8
v012                          int8
v013                          int8
v024                          int8
secoreg                       int8
v040                         int16
anemia_binary              float64
had_recent_birth             int64
antenatal_visits_filled    float64
dtype: object

In [ ]:
categorical_cols = ['v190a', 'v106', 'v116', 'v113', 'v013', 'v024', 'secoreg', 'had_recent_birth']
numeric_cols = ['v133', 'v012', 'v040', 'antenatal_visits_filled']
target_col = 'anemia_binary'

print("Categorical:", categorical_cols)
print("Numeric:", numeric_cols)
print("Target:", target_col)

Categorical: ['v190a', 'v106', 'v116', 'v113', 'v013', 'v024', 'secoreg', 'had_recent_birth']
Numeric: ['v133', 'v012', 'v040', 'antenatal_visits_filled']
Target: anemia_binary


In [ ]:
X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)
print("Train class balance:\n", y_train.value_counts(normalize=True))
print("Test class balance:\n", y_test.value_counts(normalize=True))

Train shape: (2696, 12)  Test shape: (675, 12)
Train class balance:
 anemia_binary
0.0    0.684347
1.0    0.315653
Name: proportion, dtype: float64
Test class balance:
 anemia_binary
0.0    0.684444
1.0    0.315556
Name: proportion, dtype: float64


In [ ]:
#tree_X_train = X_train.copy()
tree_X_test = X_test.copy()

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    tree_X_train[col] = le.fit_transform(tree_X_train[col].astype(str))
    tree_X_test[col] = tree_X_test[col].astype(str).map(
        lambda x: le.transform([x])[0] if x in le.classes_ else -1
    )
    encoders[col] = le

print("Tree-ready train shape:", tree_X_train.shape)
print("Tree-ready test shape:", tree_X_test.shape)

Tree-ready train shape: (2696, 12)


In [ ]:
neural_X_train = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
neural_X_test = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

# Align test columns to train FIRST — handles rare categories present in one split but not the other
neural_X_test = neural_X_test.reindex(columns=neural_X_train.columns, fill_value=0)

# Cast bool dummy columns to int (avoids SHAP/rint errors downstream)
neural_X_train = neural_X_train.astype({col: int for col in neural_X_train.select_dtypes('bool').columns})
neural_X_test = neural_X_test.astype({col: int for col in neural_X_test.select_dtypes('bool').columns})

# Scale numeric columns, fit on train only
scaler = StandardScaler()
neural_X_train[numeric_cols] = scaler.fit_transform(neural_X_train[numeric_cols])
neural_X_test[numeric_cols] = scaler.transform(neural_X_test[numeric_cols])

print("Neural-ready train shape:", neural_X_train.shape)
print("Neural-ready test shape:", neural_X_test.shape)

# Sanity check — this MUST print True, or downstream models will fail
print("Columns match:", list(neural_X_train.columns) == list(neural_X_test.columns))

In [6]:
neural_X_train = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
neural_X_test = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

# Cast bool dummy columns to int — avoids downstream issues with SHAP and some sklearn/DL operations
neural_X_train = neural_X_train.astype({col: int for col in neural_X_train.select_dtypes('bool').columns})
neural_X_test = neural_X_test.astype({col: int for col in neural_X_test.select_dtypes('bool').columns})

In [7]:
# Tree-ready
tree_X_train.to_pickle('../data/processed/tree_ready/X_train.pkl')
tree_X_test.to_pickle('../data/processed/tree_ready/X_test.pkl')
y_train.to_pickle('../data/processed/tree_ready/y_train.pkl')
y_test.to_pickle('../data/processed/tree_ready/y_test.pkl')

# Neural-ready
neural_X_train.to_pickle('../data/processed/neural_ready/X_train.pkl')
neural_X_test.to_pickle('../data/processed/neural_ready/X_test.pkl')
y_train.to_pickle('../data/processed/neural_ready/y_train.pkl')
y_test.to_pickle('../data/processed/neural_ready/y_test.pkl')

# Save encoders/scaler for later interpretability + consistent inference
with open('../data/processed/tree_ready/label_encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)
with open('../data/processed/neural_ready/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("All train/test sets and encoders saved.")

All train/test sets and encoders saved.
